In [ ]:
# -*- coding: utf-8 -*-
"""
Project Python 8.ipynb
BASF SE, a global leader in chemical manufacturing, is seeking to enhance environmental monitoring capabilities by predicting the Air Quality Index (AQI) in industrial and urban areas. Using historical air quality datasets containing parameters such as PM2.5, PM10, NO₂, SO₂, CO, O₃, temperature, and humidity, students will develop a supervised regression model to predict future AQI values.The goal is to assist environmental engineers and regulatory bodies in taking proactive measures to control emissions and safeguard public health.
"""

# -*- coding: utf-8 -*-
"""Air Quality Index Prediction - Cleaned Script with Direct CSV Upload (No Kaggle API)"""

# ==========================
# 1. Imports & Setup
# ==========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from google.colab import files

sns.set_style("whitegrid")
np.random.seed(42)

# ==========================
# 2. Upload & Load CSV Dataset
# ==========================
print("--- Data Loading ---")
print("Please upload your dataset CSV file:")

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

df = pd.read_csv(csv_filename)

if df.empty:
    raise ValueError("Error: Loaded dataset is empty.")

# Ensure non-negative values
print(f"Dataset loaded. Shape: {df.shape}")
display(df.head())

# ==========================
# 3. Data Preprocessing
# ==========================

print("--- Data Preprocessing ---")

# Define the target column
# You can change 'PM2.5 (µg/m³)' to any other column name in your dataset
target_column = 'PM2.5'

if target_column not in df.columns:
    raise ValueError(f"Error: Target column '{target_column}' not found in the dataset.")

# Check for missing values
missing = df.isna().sum()
print('Missing values in each column:')
print(missing)

# If missing values are found, we need to handle them. Here we perform a simple median imputation for numeric columns.
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

# It is important to handle missing values before modeling to avoid errors like those encountered by many notebook creators

# Double check that there are no NaNs remaining in numeric columns
print('\nAfter imputation, missing values in numeric columns:')
print(df[numeric_cols].isna().sum())



# Reset index after dropping rows if needed
df.reset_index(drop=True, inplace=True)
# Dynamically get features from the dataframe, excluding the target
features = [col for col in df.columns if col != target_column]

X = df[features]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split and scaled.")

# ==========================
# 4. Model Training
# ==========================
print("--- Model Training ---")

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

# Save model & scaler
joblib.dump(model, "rf_aqi_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved.")

# ==========================
# 5. Model Evaluation
# ==========================
print("--- Model Evaluation ---")

y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

# ==========================
# 6. Visualization
# ==========================
print("--- Visualization ---")

# Plot actual vs predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel(f"Actual {target_column}")
plt.ylabel(f"Predicted {target_column}")
plt.title(f"Actual vs Predicted ({target_column})")
plt.show()

# Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True)
plt.title(f"Residuals Distribution ({target_column})")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.show()


# ==========================
# 7. Exploratory Plots
# ==========================
print("--- Exploratory Visualizations ---")

numerical_features = features + [target_column] # Include target for exploratory plots
plt.figure(figsize=(15, 10))
# Plot up to 9 numerical features in a 3x3 grid
for i, feature in enumerate(numerical_features[:9]):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
plt.tight_layout()
plt.show()

# Select a few features for scatter plots if there are many
if len(features) > 3:
    sample_features_for_scatter = feature_importances.head(3).index.tolist() if 'feature_importances' in locals() else features[:3]
else:
    sample_features_for_scatter = features

# Scatter plots against the placeholder target
plt.figure(figsize=(15, 5))
for i, feature in enumerate(sample_features_for_scatter):
    plt.subplot(1, len(sample_features_for_scatter), i+1)
    plt.scatter(df[feature], df[target_column], alpha=0.5)
    plt.xlabel(feature)
    plt.ylabel(target_column)
    plt.title(f'{feature} vs {target_column}')
plt.tight_layout()
plt.show()


plt.plot(df.index, df[target_column])
plt.title(f'Raw {target_column} Values')
plt.xlabel('Index')
plt.ylabel(target_column)
plt.show()

# ==========================
# 8. Predictions on New Data
# ==========================
print("--- Prediction on New Data ---")

loaded_model = joblib.load("rf_aqi_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

# Update sample data to match the new feature set
sample_data = pd.DataFrame({feature: [10, 20, 5] for feature in features}) # Placeholder values for features
# Add a placeholder for the target column if needed for the dataframe structure, but it won't be used for prediction input
sample_data[target_column] = [0, 0, 0] # Placeholder target values

sample_scaled = loaded_scaler.transform(sample_data[features]) # Scale only the features
predictions = loaded_model.predict(sample_scaled)

print("Sample Data:")
display(sample_data)
print(f"\nPredicted Values ({target_column}):")
for i, val in enumerate(predictions):
    print(f"Sample {i+1}: {val:.2f}")